# Silver layer — clean, typed, de-duplicated

Reads the bronze raw tables and produces silver tables with proper types, trimmed keys and one table per feed. The `pos_sales` feed additionally gets the full data-quality treatment:

* **Exact duplicate lines** are de-duplicated; the extra copies are quarantined.
* **Barcodes missing from the product master** are quarantined.
* **Store codes with stray whitespace** are trimmed (and any code still unknown is quarantined).
* **Missing discounts** stay `NULL` and are flagged with `discount_missing`.
* **Returns (negative quantity) are real business events** — they are kept, never quarantined.

Rejected rows land in `pos_sales_quarantine` with the rule they failed, so that bronze rows = silver rows + quarantined rows reconciles exactly.

In [0]:
import re

from pyspark.sql import functions as F


def _param(name, default):
    """Widget value if defined, otherwise the default, so the notebook also runs headless."""
    try:
        value = dbutils.widgets.get(name).strip()
        return value or default
    except Exception:
        return default


CATALOG = _param("catalog", "genie_agents_poc")
BRONZE = _param("bronze_schema", "bronze")
SILVER = _param("silver_schema", "silver")
GOLD = _param("gold_schema", "gold")

_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for label, value in [("catalog", CATALOG), ("bronze_schema", BRONZE),
                     ("silver_schema", SILVER), ("gold_schema", GOLD)]:
    if not _IDENT.match(value):
        raise ValueError(f"{label} may contain only letters, digits and underscores; got {value!r}")


def bronze(feed):
    """Business columns of a bronze raw table, without the ingestion metadata."""
    return spark.table(f"{CATALOG}.{BRONZE}.{feed}_raw").drop("_source_feed", "_source_file", "_ingested_at")


def typed(df, dates=(), ints=(), doubles=(), bools=()):
    """Trim every string column and cast the listed columns to their types."""
    casts = {}
    for cols, t in [(dates, "date"), (ints, "int"), (doubles, "double"), (bools, "boolean")]:
        for c in cols:
            casts[c] = t
    return df.select([
        (F.trim(F.col(c)).cast(casts[c]) if c in casts else F.trim(F.col(c))).alias(c)
        if dtype == "string" else F.col(c)
        for c, dtype in df.dtypes])


def save_silver(df, table, grain):
    target = f"{CATALOG}.{SILVER}.{table}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    spark.sql(f"COMMENT ON TABLE {target} IS '{grain}'")
    print(f"  {table:<26}{spark.table(target).count():>10,} rows -> {target}")

In [0]:
# Simple feeds: cast types, trim keys, no data-quality issues expected in these.
save_silver(typed(bronze("stores")), "stores",
            "Silver stores - one row per store. Keys trimmed, no other changes.")
save_silver(typed(bronze("products"), doubles=["mrp"], bools=["is_pivotal"]), "products",
            "Silver products - one row per product (EAN = style x colour x size).")
save_silver(typed(bronze("warehouse_stock_daily"), dates=["snapshot_date"], ints=["available_qty"]),
            "warehouse_stock_daily",
            "Silver warehouse stock - one row per day x warehouse x EAN.")
save_silver(typed(bronze("store_stock"), dates=["snapshot_date"], ints=["stock_on_hand"]),
            "store_stock",
            "Silver store stock - one row per store x EAN, snapshot on the as-of date.")
save_silver(typed(bronze("in_transit"), dates=["dispatch_date"], ints=["qty"]),
            "in_transit",
            "Silver in-transit - one row per transfer dispatched to a store, not yet received.")
save_silver(typed(bronze("style_holds"), dates=["hold_from"]),
            "style_holds",
            "Silver style holds - one row per style blocked from replenishment.")
save_silver(typed(bronze("clearance_styles"), dates=["clearance_from"]),
            "clearance_styles",
            "Silver clearance styles - one row per end-of-season style.")
save_silver(typed(bronze("store_capacity"), ints=["max_units"]),
            "store_capacity",
            "Silver store capacity - one row per store x category.")
save_silver(typed(bronze("depth_targets"), ints=["depth_target"]),
            "depth_targets",
            "Silver depth targets - one row per store grade x size class.")
save_silver(typed(bronze("replenishment_decisions"), dates=["run_date"], ints=["units_needed", "order_qty"]),
            "replenishment_decisions",
            "Silver replenishment decisions - one row per store x EAN, engine output for the day.")

  stores                            40 rows -> genie_agents_poc.silver.stores
  products                         440 rows -> genie_agents_poc.silver.products
  warehouse_stock_daily         79,200 rows -> genie_agents_poc.silver.warehouse_stock_daily
  store_stock                   11,199 rows -> genie_agents_poc.silver.store_stock
  in_transit                       791 rows -> genie_agents_poc.silver.in_transit
  style_holds                        3 rows -> genie_agents_poc.silver.style_holds
  clearance_styles                   6 rows -> genie_agents_poc.silver.clearance_styles
  store_capacity                   186 rows -> genie_agents_poc.silver.store_capacity
  depth_targets                      6 rows -> genie_agents_poc.silver.depth_targets
  replenishment_decisions       11,199 rows -> genie_agents_poc.silver.replenishment_decisions


In [0]:
from pyspark.sql import Window

raw_pos = bronze("pos_sales")
POS_COLS = [c for c, _ in raw_pos.dtypes]
pos = typed(raw_pos, dates=["bill_date"], ints=["qty", "discount_pct"], doubles=["mrp", "net_amount"])

# How many rows arrived with stray whitespace on the store code (fixed by the trim above).
ws_rows = raw_pos.filter(F.col("store_code") != F.trim(F.col("store_code"))).count()

# Rule 1: exact duplicate transaction lines - keep the first copy, quarantine the rest.
pos = pos.withColumn("_rn", F.row_number().over(Window.partitionBy(*POS_COLS).orderBy("txn_id")))
dup_rows = pos.filter(F.col("_rn") > 1).drop("_rn")
pos = pos.filter(F.col("_rn") == 1).drop("_rn")

# Rule 2: barcode missing from the product master.
products = spark.table(f"{CATALOG}.{SILVER}.products").select("ean")
unknown_product = pos.join(products, "ean", "left_anti")
pos = pos.join(products, "ean", "left_semi")

# Rule 3: store code not found in the stores master (after trimming).
stores = spark.table(f"{CATALOG}.{SILVER}.stores").select("store_code")
unknown_store = pos.join(stores, "store_code", "left_anti")
pos = pos.join(stores, "store_code", "left_semi")


def _quarantined(df, rule, detail):
    return (df.withColumn("rule", F.lit(rule))
              .withColumn("detail", F.lit(detail))
              .withColumn("_quarantined_at", F.current_timestamp()))


quarantine = (
    _quarantined(dup_rows, "DUPLICATE_ROW", "Exact duplicate of a kept transaction line; first copy kept")
    .unionByName(_quarantined(unknown_product, "UNKNOWN_PRODUCT", "EAN missing from the product master"))
    .unionByName(_quarantined(unknown_store, "UNKNOWN_STORE", "Store code not found in the stores master"))
)
save_silver(quarantine, "pos_sales_quarantine",
            "Silver pos_sales quarantine - one row per rejected transaction line, with the rule it failed.")

# Returns are real business events: keep them, flagged. Missing discounts stay NULL, flagged.
silver_pos = (
    pos.withColumn("is_return", F.col("qty") < 0)
       .withColumn("discount_missing", F.col("discount_pct").isNull())
       .select("txn_id", "bill_date", "store_code", "ean", "qty", "is_return", "mrp",
               "discount_pct", "discount_missing", "net_amount")
)
save_silver(silver_pos, "pos_sales",
            "Silver POS sales - one row per transaction line (store x product x day). "
            "Negative qty is a genuine return and is kept; rows failing a data-quality rule "
            "are in pos_sales_quarantine, not here. Missing discount_pct stays NULL with "
            "discount_missing = true; net_amount comes from the source feed.")

# Reconciliation: bronze = silver + quarantine.
b, s = raw_pos.count(), silver_pos.count()
by_rule = {r["rule"]: r["n"] for r in quarantine.groupBy("rule").agg(F.count("*").alias("n")).collect()}
returns = silver_pos.filter("is_return").count()
print("\npos_sales reconciliation")
print(f"  bronze rows                {b:>10,}")
print(f"  silver rows                {s:>10,}")
for rule in ["DUPLICATE_ROW", "UNKNOWN_PRODUCT", "UNKNOWN_STORE"]:
    print(f"  quarantined {rule:<15}{by_rule.get(rule, 0):>6,}")
print(f"  store codes with stray whitespace (trimmed): {ws_rows:,}")
print(f"  returns kept in silver: {returns:,} (returns are not a data-quality issue)")
assert b == s + sum(by_rule.values()), "bronze rows != silver + quarantined"
print("  OK: bronze = silver + quarantined")

  pos_sales_quarantine           1,102 rows -> genie_agents_poc.silver.pos_sales_quarantine
  pos_sales                    135,067 rows -> genie_agents_poc.silver.pos_sales

pos_sales reconciliation
  bronze rows                   136,169
  silver rows                   135,067
  quarantined DUPLICATE_ROW     703
  quarantined UNKNOWN_PRODUCT   399
  quarantined UNKNOWN_STORE       0
  store codes with stray whitespace (trimmed): 0
  returns kept in silver: 3,962 (returns are not a data-quality issue)
  OK: bronze = silver + quarantined


In [0]:
display(spark.table(f"{CATALOG}.{SILVER}.pos_sales").orderBy("bill_date", "store_code").limit(5))

txn_id,bill_date,store_code,ean,qty,is_return,mrp,discount_pct,discount_missing,net_amount
TX20260718-S1001-1eeb8ac1a7,2026-07-18,S1001,8900000000209,1,false,1099.0,10,false,989.1
TX20260718-S1001-1e4dde6e9e,2026-07-18,S1001,8900000000211,1,false,1099.0,0,false,1099.0
TX20260718-S1001-0742d69b58,2026-07-18,S1001,8900000000092,1,false,2499.0,10,false,2249.1
TX20260718-S1001-1e8080f6a9,2026-07-18,S1001,8900000000083,2,false,1599.0,0,false,3198.0
TX20260718-S1001-25f4edf937,2026-07-18,S1001,8900000000297,1,false,2199.0,0,false,2199.0
